In [ ]:
# !pip install gtts
# !git clone https://github.com/VarunGumma/IndicTransToolkit
# !cd /Users/kpkautum/Desktop/PJT2/IndicTransToolkit/ && python3 -m pip install --editable .

In [ ]:
import torch
from transformers import BlipProcessor, BlipForConditionalGeneration, Blip2ForConditionalGeneration, Blip2Processor
from gtts import gTTS
import os
from PIL import Image
from huggingface_hub import hf_hub_download
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from IndicTransToolkit.IndicTransToolkit import IndicProcessor


def generate_caption(image_path):
    model_name = "Salesforce/blip2-flan-t5-xl"
    model = Blip2ForConditionalGeneration.from_pretrained(model_name)
    processor = Blip2Processor.from_pretrained(model_name)

    image = Image.open(image_path).convert("RGB")

    text_prompt = "Describe the content in the image."

    inputs = processor(images=image, text=text_prompt, return_tensors="pt")

    outputs = model.generate(
        **inputs,
        max_length=300,
        num_beams=5,
        length_penalty=2.0,
        repetition_penalty=2.0
    )

    response = processor.tokenizer.decode(outputs[0], skip_special_tokens=True)

    print("Generated Response:")
    print(response)
    return response

def generate_caption_for_face(image_path):
    repo_id = "Duke29/Face_Finetuned_Salesforce_Blip_Image_Captioning_Base"
    model_filename = "finetuned_blip_captioning.pth"
    device = "cuda" if torch.cuda.is_available() else "cpu"

    print(f"Using device: {device}")

    processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
    model_path = hf_hub_download(repo_id=repo_id, filename=model_filename)

    # Load the model architecture
    model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device)

    # Load the fine-tuned weights
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    image = Image.open(image_path).convert("RGB")
    inputs = processor(images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(**inputs, output_scores=True, return_dict_in_generate=True)

    caption = processor.tokenizer.decode(outputs.sequences[0], skip_special_tokens=True)

    # Calculate confidence score
    probs = torch.nn.functional.softmax(outputs.scores[0], dim=-1)
    confidence_score = torch.max(probs).item() * 100

    print("Generated Caption:", caption)
    print(f"Confidence Score: {confidence_score:.2f}%")

    return caption, confidence_score

def translate_caption(text, target_language="hi"):

    if target_language == "hi":
        tgt_lang = "hin_Deva"
    elif target_language == "ta":
        tgt_lang = "tam_Taml"
    elif target_language == "te":
        tgt_lang = "tel_Telu"
    src_lang = "eng_Latn"

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    model_name = "ai4bharat/indictrans2-en-indic-1B"
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

    model = AutoModelForSeq2SeqLM.from_pretrained(
        model_name,
        trust_remote_code=True,
        torch_dtype=torch.float16,
        attn_implementation="flash_attention_2"
    ).to(DEVICE)

    ip = IndicProcessor(inference=True)

    input_sentences = [text]

    batch = ip.preprocess_batch(
        input_sentences,
        src_lang=src_lang,
        tgt_lang=tgt_lang,
    )

    inputs = tokenizer(
        batch,
        truncation=True,
        padding="longest",
        return_tensors="pt",
        return_attention_mask=True,
    ).to(DEVICE)

    with torch.no_grad():
        generated_tokens = model.generate(
            **inputs,
            use_cache=True,
            min_length=0,
            max_length=256,
            num_beams=5,
            num_return_sequences=1,
        )

    with tokenizer.as_target_tokenizer():
        generated_tokens = tokenizer.batch_decode(
            generated_tokens.detach().cpu().tolist(),
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        )

    translations = ip.postprocess_batch(generated_tokens, lang=tgt_lang)

    for input_sentence, translation in zip(input_sentences, translations):
        print(f"{src_lang}: {input_sentence}")
        print(f"{tgt_lang}: {translation}")
    return translation

def text_to_speech(text, filename, language="en"):
    tts = gTTS(text=text, lang=language)
    tts.save(filename)
    #os.system(f"mpg321 {filename}")

def process_image(image_path, target_language):
    caption = generate_caption(image_path)
    print("BLIP-2 Caption:", caption)

    if any(word in caption.lower() for word in ["man", "woman", "child"]):
        custom_caption, confidence_score = generate_caption_for_face(image_path)

        if confidence_score >= 90:
            print("Using Custom BLIP Caption")
            caption = custom_caption
        else:
            print("Low Confidence, Using BLIP-2 Caption")

    translated_caption = translate_caption(caption, target_language)
    print("Translated Caption:", translated_caption)

    text_to_speech(caption, "caption_en.mp3")
    text_to_speech(translated_caption, "caption_translated.mp3", language=target_language)


image_path = "/kaggle/input/pathway-image/pathway.jpg"
target_language = "hi"

process_image(image_path, target_language)


In [ ]:
image_path = "/Users/kpkautum/Desktop/PJT2/360_F_649400293_TaWgud2YIPIXlGZsy2cFlM7bsU6C19yv.jpg"
target_language = "ta"

process_image(image_path, target_language)

In [ ]:
from IPython.display import FileLinks
# Generate links for all files in the current directory
FileLinks(".")
